In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import pickle
import config


In [ ]:
from sentence_transformers import CrossEncoder,InputExample
from torch.utils.data import DataLoader
from src.metric import model_evaluation


In [ ]:
path=config.CLEANED_DATA_DIR

In [ ]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [ ]:
BATCH_SIZE=config.BASELINE_BATCH_SIZE

In [ ]:

cross_train_examples=[
    InputExample(texts=[j,r],label=float(config.label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

In [ ]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=BATCH_SIZE)

In [ ]:
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=config.device)

In [ ]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


In [ ]:
model_save_path=os.path.join(config.BASELINE_MODEL_DIR,'cross_encoder')
os.makedirs(config.BASELINE_MODEL_DIR,exist_ok=True)

In [ ]:
epochs=4
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score = 0.6*metrics['ndcg_val']+0.3*metrics['map_score']+0.1*metrics['mrr_score']
    
    if final_score>best_score + min_delta:
        best_score=final_score
        cross_encoder_model.save(model_save_path)
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


In [ ]:
cross_encoder_model=CrossEncoder(model_save_path,device=config.device)

In [ ]:
val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)


In [ ]:
metrics=model_evaluation(scores,val_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

In [ ]:
ranked_result=[]
eval_df=val_df.copy()
eval_df['score']=scores

In [ ]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")

In [ ]:
for jd,group in eval_df.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [ ]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

In [ ]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])
